# Show how nucleosome entropy is related to replication and gene expression level

May 14, 2024

**Goal:** Discern how nucleosome entropy is affected by:
1. Replication timing: early vs late
2. Gene expression, level and regulation


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.figure_configs import save_figure_for_paper, FiguresConfig


ModuleNotFoundError: No module named 'src'

In [ ]:
from src.global_config import GlobalConstants

repl_timing = pd.read_csv(GlobalConstants.REPL_TIMING_FILEPATH).set_index('orf_name')
print(len(repl_timing), " genes with replication profiles")
repl_timing.head()

In [ ]:
from src.promoter_ptr_analysis import PromoterPTRAnalysis

chromatin_dir = 'output/deconvolve_combined_g_opt_2024_03_14/chromatin/'
promoter_analysis = PromoterPTRAnalysis(chromatin_dir)
promoter_analysis.load_f_files()

In [ ]:
promoter_analysis.correct_f_images_by_strand()

In [ ]:
# Let's get the gene expression at the replication time.

from src.gene_expression_deconv_analysis import GeneExpressionAnalysis

ge_analysis = GeneExpressionAnalysis(
    'output/deconvolve_combined_g_opt_2024_03_14/gene_expression/')
ge_analysis.load_gene_expression_fs()


In [ ]:
from src.gene_chromatin_replication_analysis import GeneChromatinReplicationAnalysis

gene_chrom_repl_analysis = GeneChromatinReplicationAnalysis(promoter_analysis,
                                                            ge_analysis,
                                                            repl_timing)

In [ ]:
from src.helpers import calc_entropy

In [ ]:
gene_chrom_repl_analysis.compute_gene_nuc_entropy()

In [ ]:
gene_chrom_repl_analysis.normalized_entropy_df.to_csv('output/normalized_gb_entropy.csv')

In [ ]:
tx_repl = gene_chrom_repl_analysis.geneset_repl.expression_at_replication
plt.figure(figsize=(4, 2))
plt.hist(tx_repl, bins=60)
plt.title("Gene expression at replication time")
plt.xlabel("Expression, VST")

q25, q75 = np.quantile(tx_repl, q=[0.1, 0.9])

plt.axvline(q25, c='red')
plt.axvline(q75, c='red')

np.sum(tx_repl > q75), np.sum(tx_repl <  q25)

high_tx_orfs = tx_repl.loc[tx_repl > q75].index
low_tx_orfs = tx_repl.loc[tx_repl < q25].index

In [ ]:
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision(normalize=True)

In [ ]:
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision(normalize=True,
    subset_orfs=high_tx_orfs, title="Highest transcribed genes, entropy")
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision(normalize=True,
    subset_orfs=low_tx_orfs, title="Lowest transcribed genes, entropy")


In [ ]:
# In the unnormalized entropy scores, can we see a global effect of the replication fork
# passing through early and late genes?
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision()


In [ ]:
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision(normalize=False,
    subset_orfs=high_tx_orfs, title="Highest transcribed genes, entropy")
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision(normalize=False,
    subset_orfs=low_tx_orfs, title="Lowest transcribed genes, entropy")


# Analysis to work through absolute entropy values

Absolute entropy values make more sense and help us have an idea on the overall entropy across the genome. But the values span and oscillate along different entropy values making it difficult to identify patterns in oscillation for early vs late replicating genes without normalization.

If we want to avoid normalization, which may add an additional layer of complexity, we can try to break the entropy scores in to quantiles that make it easier to compare early and late replicating genes. 

**Idea:** Gene nucleosomes have different baseline levels of entropy (some sets of nucleosomes shift more than others). So let's split them up. Perhaps these are nucleosomes that more commonly pickup certain histone variants (possible side analysis).

**Hypothesis**: Genes with nucleosomes that are more entropic overall, have a differential timing of peak entropy alongside replication than nucleosomes with less entropy.

In [ ]:
gene_chrom_repl_analysis.set_entropy_vs_tx_df(mode='at_replication')
gene_chrom_repl_analysis.plot_early_late_entropy_hm_comparision(normalize=True)

In [ ]:
gene_chrom_repl_analysis.plot_hi_lo_entropy_hm_analysis()

There appears to be some differences in replication timing depending on the average entropy of the gene nucleosomes.

If we were to examining these genes by, early genes and late genes. How would this pattern change?
What is the question?

**Hypothesis:** High entropic genes, have a higher baseline level of transcription and are less likely to be affected by the replication fork disruption.

In [ ]:
# For the previous analysis it made sense to use mean entropy because we are 
# plotting the whole  time course

# But here, we are plotting the expresison at replication time
# so it would be logical to use the entropy value at replication time
x_cutoffs = [9, 12]

# Snap shot of genomic and chromatin landscape at replication time:
gene_chrom_repl_analysis.plot_entropy_vs_expression_analysis(x_cutoffs=x_cutoffs)


Because this plot shows mean entropy, the hypothesis cannot be concluded.

But interesting observation from this series:
1. Early replicating genes have differential expression and entropy for high and lowly expressed genes.
2. High and low expressed genes (possibly regulated), tend to have higher average entropy values in early replicating genes compared to late replicating gene.
 - Maybe there a regulatory implications here.

In [ ]:
gene_chrom_repl_analysis.plot_box_plot_entropy_ge(x_cutoffs)

# Reflect: Biological implications

Why is the chromatin for early replicating genes with high transcription more entropic, than late replicating genes with high transcription?

Hypothesis:
- Late replicating genes must be more organized in order to handle late replication stress
- Early replicating genes are more adaptable, to allow time for replication repair.

How can we provide evidence for this hypothesis? What genes are in these two groups and why? What genes are activated by the time these genes are being replicated?


# Plan: Compute the average entropy and gene expression values through replication. Delta of 5 and 10 minutes

    
Goal: Compute metrics based on range rather than replication point to reduce variation in replication timing estimation.

Redo the analysis of early and late replication with the average delta values and observe if the result stands, and/or if there are other observations made when the delta range is in effect.

Try for 5 and 10 minutes. How robust are these results? How much does increasing the average window affect the results? We should expect that +/- 10 minutes (a 20 minute) window should reduce the signal. And +/- 5 minutes (10 minute window) should reduce noise.


In [ ]:
gene_chrom_repl_analysis.compute_replication_window_deltas()

In [ ]:
gene_chrom_repl_analysis.plot_replication_delta_windows()

In [ ]:
# Compute the entropy and gene expression 
# averaged across the replication timing delta values
#
# Compute the -delta and +delta values
gene_chrom_repl_analysis.compute_delta_entropies()
gene_chrom_repl_analysis.compute_delta_gene_expressions()

In [ ]:
x_cutoffs = [7, 14]

In [ ]:
gene_chrom_repl_analysis.set_entropy_vs_tx_df(mode='avg_delta_5_replication')
gene_chrom_repl_analysis.plot_hi_lo_entropy_hm_analysis()

In [ ]:
gene_chrom_repl_analysis.plot_entropy_vs_expression_analysis(x_cutoffs=x_cutoffs)
gene_chrom_repl_analysis.plot_box_plot_entropy_ge(x_cutoffs)

**Reflections:** This story shows a snapshot of the entropy. It, shows the difference in entropy at replication time for highly transcribed genes.

But does not say anything yet about the cycling nature of the expression nor entropy. How can we further advance this story to get a sense of replication for transcriptional regulation?

**Goal:** Discern: 
- Are the entropy values and expression dynamic/cycling for thse genes?
    - If so, are they at replication time or elsewhere?
        - There can be regulational implications for either case.
- Can the heatmap of the entire timecourse help with this question based on careful segmentation of the gene sets?

**Possible tasks:**
- Create a heatmap of the time course for entropy separated by gene expression.
- How does the direction of gene transcription play a role in this analysis?
    - Is this the time to start looking at those gene segments?
    

In [ ]:
# Get the group cutoffs for tx during replication
gene_entropy_tx_w_tx_cutoffs = gene_chrom_repl_analysis.ge_get_cutoffs_at_repl(x_cutoffs)
gene_entropy_tx_w_tx_cutoffs = gene_entropy_tx_w_tx_cutoffs.sort_values('replication_timing')

# Compute the PTR values for entropy
from src.peak_to_trough import compute_quantile_ptr

nuc_entropy_ptrs = np.apply_along_axis(compute_quantile_ptr, 1, nuc_entropy.fillna(0).values)
nuc_entropy_ptrs = pd.Series(nuc_entropy_ptrs, index=nuc_entropy.index)

# Append onto metadata for entropy and expresison table
gene_entropy_tx_w_tx_cutoffs['entropy_ptr'] = nuc_entropy_ptrs

In [ ]:
# Plot the heatmap of the entropy and expression for these gene groups
# We will care most about the high expression group.

nuc_entropy = gene_chrom_repl_analysis.entropy_df

tx_groups = gene_entropy_tx_w_tx_cutoffs.group_name.unique()

# High expression genes at replication time
hi_tx_group = tx_groups[-1]
hi_tx_genes = gene_entropy_tx_w_tx_cutoffs[\
    gene_entropy_tx_w_tx_cutoffs.group_name == hi_tx_group]

lo_tx_group = tx_groups[0]
lo_tx_genes = gene_entropy_tx_w_tx_cutoffs[\
    gene_entropy_tx_w_tx_cutoffs.group_name == lo_tx_group]

def plot_hm(dat, vmin=2, vmax=4, cmap='viridis'):
    plt.imshow(dat, aspect='auto', vmin=vmin, vmax=vmax, cmap=cmap)

# Let's plot the heatmap of the entropy in the time course
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
dat = nuc_entropy.loc[lo_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, lo_tx_genes, vmin=2, vmax=4,
                                     title="Low transcription", 
                                      fig=fig, cmap='Spectral')
plt.colorbar()

plt.subplot(1, 2, 2)
dat = nuc_entropy.loc[hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, hi_tx_genes, vmin=2, vmax=4,
                                     title="High transcription", 
                                      fig=fig, cmap='Spectral')
plt.colorbar()
plt.suptitle("Entropy values separated by expression during replication")

# It is difficult to discern patterns because different genes appear to have different
# basal levels of expression. (value of change in entropy model)

# Our story is based around high transcription genes at replication
# So let's drill into those genes and see if we can see the entropy pattern
# among early and late genes, we may need to split up the entropy values
# to actually see the patterns.

In [ ]:
# What are appropriate cutoff values that would discern the early and late tx genes?
# Refer back to the scatter plot. The early and late replicated genes
# appear to be separated by these cutoffs
entropy_cutoffs = 3.3, 3.6
plt.figure(figsize=(4, 2))
plt.hist(gene_entropy_tx_w_tx_cutoffs.entropy_value, bins=30)
for e in entropy_cutoffs:
    plt.axvline(e, c='red')
plt.title("Distribution of entropy values during replication")

In [ ]:
low_e_hi_tx_genes = hi_tx_genes[hi_tx_genes.entropy_value < entropy_cutoffs[0]]
hi_e_hi_tx_genes = hi_tx_genes[hi_tx_genes.entropy_value > entropy_cutoffs[1]]

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
dat = nuc_entropy.loc[low_e_hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, low_e_hi_tx_genes, vmin=2, vmax=4,
                                     title="Low entropy", 
                                      fig=fig, cmap='Spectral')
plt.colorbar()

plt.subplot(1, 2, 2)
dat = nuc_entropy.loc[hi_e_hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, hi_e_hi_tx_genes, vmin=2, vmax=4,
                                     title="High entropy", 
                                      fig=fig, cmap='Spectral')
plt.colorbar()
plt.suptitle("High expression genes during replication")

**Reflect**: Cycling behavior is not prevalent in these segments

**Plan**: Segment the genes by cycling and non-cycling entropy and plot. We hope to see a pattern/discern how many genes are cycling and where they are peaking/troughing. Compute the PTRs and split the plot into quadrants

In [ ]:
plt.figure(figsize=(4, 2))
plt.hist(nuc_entropy_ptrs, bins=50)
plt.title("Entropy PTR scores")
0

In [ ]:
lo_e_hi_tx_genes = hi_tx_genes[(hi_tx_genes.entropy_value < entropy_cutoffs[0]) & 
                               (hi_tx_genes.entropy_value > 3)]
hi_e_hi_tx_genes = hi_tx_genes[hi_tx_genes.entropy_value > entropy_cutoffs[1]]

cycling_threshold = 1.025

lo_e_cc_hi_tx_genes = lo_e_hi_tx_genes[lo_e_hi_tx_genes.entropy_ptr > cycling_threshold]
lo_e_noncc_hi_tx_genes = lo_e_hi_tx_genes[lo_e_hi_tx_genes.entropy_ptr < cycling_threshold]

hi_e_cc_hi_tx_genes = hi_e_hi_tx_genes[hi_e_hi_tx_genes.entropy_ptr > cycling_threshold]
hi_e_noncc_hi_tx_genes = hi_e_hi_tx_genes[hi_e_hi_tx_genes.entropy_ptr < cycling_threshold]

fig = plt.figure(figsize=(8, 9))
plt.subplot(2, 2, 1)
dat = nuc_entropy.loc[lo_e_cc_hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, lo_e_cc_hi_tx_genes, vmin=2, vmax=4,
                                     title="Low entropy cycling", fig=fig, cmap='Spectral')
#plot_hm(dat, cmap='Spectral')
plt.colorbar()

plt.subplot(2, 2, 2)
dat = nuc_entropy.loc[hi_e_cc_hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, hi_e_cc_hi_tx_genes, vmin=2, vmax=4,
                                     title="High entropy cycling", fig=fig, cmap='Spectral')
plt.colorbar()
plt.title(f"High entropy, cycling, n={len(dat)}")

plt.subplot(2, 2, 3)
dat = nuc_entropy.loc[lo_e_noncc_hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values, 
                                      lo_e_noncc_hi_tx_genes, vmin=2, vmax=4,
                                     title="Low entropy non-cycling", fig=fig, cmap='Spectral')
plt.colorbar()

plt.subplot(2, 2, 4)
dat = nuc_entropy.loc[hi_e_noncc_hi_tx_genes.index]
gene_chrom_repl_analysis.plot_heatmap(dat.values,
                                      hi_e_noncc_hi_tx_genes, vmin=2, vmax=4,
                                     title="High entropy non-cycling", 
                                      fig=fig, cmap='Spectral')
plt.colorbar()

plt.suptitle("High expression genes\nEntropy segmented by entropy during"
    "\nreplication and cycling/non-cycling")

**Reflect** 
This plots are still difficult do discern a story. It shows that there is cycling around replication for high entropy early replicating genes. But late replicating is less discernable. And not very discernable for low entropy genes. Although these genes have a wider range as well, so the patterns are difficult to see.

So maybe the most important thing to observe here is the set of genes that are early replicating and high entropy that are cycling.



**Plan**: Switching gears, let's look at the early and late replicating genes. (High Tx). If we plot the entropy values across the time course as a box plot, we may be able to see the cyclical nature of the entropy scores and where they peak. From the plot above, we should expect early replicating gnes to show peak entropy early. The challenge here may be the range of entropy scores and the basal/unchanging scores washing out the signal. So we may need to separate those genes out similar to above. Same with the low and high entropy scores.



In [ ]:
# Develop:

# Get the early and late replicating gene dilineations


In [ ]:
iqr = np.quantile(nuc_entropy.fillna(0), q=[0.25, 0.5, 0.75], axis=0)
iqr.shape

In [ ]:
from src.nuc_entropy_replication_analysis import plot_metric_timecourse_by_repl

plot_metric_timecourse_by_repl(nuc_entropy, gene_chrom_repl_analysis,
                                   (3, 3.75), "Nucleosome entropy across cell cycle")

In [ ]:
t_indices = gene_chrom_repl_analysis.config.get_Hpositions_for_branch('t')
gene_expression_top_f = gene_expression_df[t_indices]
gene_expression_top_f.columns = np.arange(len(gene_expression_top_f.columns))

plot_metric_timecourse_by_repl(gene_expression_top_f, gene_chrom_repl_analysis,
                                   (9, 13), "Gene expression across cell cycle")

**Reflect:** Because there are so many genes, signal may be washed out. 

**Plan** So it may be worth looking at high and low tx genes separately next.


In [ ]:
from src.nuc_entropy_replication_analysis import plot_metric_timecourse_by_repl

plot_metric_timecourse_by_repl(nuc_entropy, gene_chrom_repl_analysis,
    (2.8, 3.9), "Low transcribed, Nucleosome entropy across cell cycle",
    tx_cutoffs=[0, 9])



In [ ]:
from src.nuc_entropy_replication_analysis import plot_metric_timecourse_by_repl

plot_metric_timecourse_by_repl(nuc_entropy, gene_chrom_repl_analysis,
    (3, 3.9), "Mid Transcribed, Nucleosome entropy across cell cycle",
    tx_cutoffs=[9, 12])



In [ ]:
from src.nuc_entropy_replication_analysis import plot_metric_timecourse_by_repl

plot_metric_timecourse_by_repl(nuc_entropy, gene_chrom_repl_analysis,
    (2.8, 3.9), "Highly Transcribed, Nucleosome entropy across cell cycle",
    tx_cutoffs=[12, 100])



**Reflect** on the Biology, what are we seeing and how does this shape the analysis. A lot of this seems to be data processing. What signal is being pulled out and makes the most sense to pursue?

**Plan** What does the timecourse look like for gene expression with these cutoffs?

In [ ]:
from src.nuc_entropy_replication_analysis import plot_metric_timecourse_by_repl

plot_metric_timecourse_by_repl(gene_expression_top_f, gene_chrom_repl_analysis,
    (5, 10), "Low transcribed, expression across cell cycle",
    tx_cutoffs=[0, 9])


In [ ]:
from src.nuc_entropy_replication_analysis import plot_metric_timecourse_by_repl

plot_metric_timecourse_by_repl(gene_expression_top_f, gene_chrom_repl_analysis,
    (12, 14.5), "High transcribed, expression across cell cycle",
    tx_cutoffs=[12, 100])

# Wider range of tx values


Identify genes with low transcription, only replication affects chromatin.

1. Genes with low transcription.
2. Genes with well-organized nucleosomes in G2M.


In [ ]:
def perform_cutoff_dat(dat, q, title, fig=None):
    
    from src.helpers import get_quantile_values
    if fig is None:
        fig = plt.figure(figsize=(4, 2))
    plt.hist(dat, bins=80)
    (low_orfs, hi_orfs), qvals, lens = get_quantile_values(dat.fillna(0), 
        [q])        
    plt.axvline(qvals[0], c='red')
    
    plt.title(f"{title}\n$q$={q*100:.0f}%, $n_q$={lens[0]}",
        fontsize=FiguresConfig.FIG_TITLE_FONTSIZE)

    return low_orfs, hi_orfs

In [ ]:
# Lowly expressed genes through the entire timecourse
fig = plt.figure(figsize=FiguresConfig.FIGSIZE_SHORT_WIDE)
plt.subplots_adjust(top=0.72)

plt.subplot(1, 2, 1)
ge_max = gene_chrom_repl_analysis.gene_expression_f.max(axis=1)
lo_max_ge_orfs, _ = perform_cutoff_dat(ge_max, .25, "Maximal gene expression", fig=fig)

g2m_indices = config.get_Hpositions_for_phase('G2/M')
g2m_indices_t = gene_chrom_repl_analysis.convert_h_indices_to_t(g2m_indices)
entropy_in_g2m = gene_chrom_repl_analysis.entropy_df[g2m_indices_t]
g2m_entropy = entropy_in_g2m.max(axis=1)

plt.subplot(1, 2, 2)
lo_entropy_orfs, _ = perform_cutoff_dat(g2m_entropy, 0.25, "Maximal entropy in G2/M",
    fig=fig)
plt.suptitle("Geneset selection for untranscribed genes", 
    fontsize=FiguresConfig.FIG_SUPTITLE_FONTSIZE)

save_figure_for_paper("output/figures/Entropy_expression_cutoffs_repl_analysis.png")

print("Genes with low transcription: ", len(lo_max_ge_orfs))
print("Genes with well-positioned nucleosomes: ", len(lo_entropy_orfs))

In [ ]:
# Merge the set of low maximal entropy (organized nucleosome genes)
# and the set of genes with low maximal gene expression

# This ensures that we have selected genes with well enough positioned nucleosomes
# that are not transcribing, that whose chromatin are clearly disrupted by replication.
lo_entropy_lo_max_ge_orfs = np.array(list(set(lo_entropy_orfs.index).intersection(
    set(lo_max_ge_orfs.index))))
len(lo_entropy_lo_max_ge_orfs)

In [ ]:
# Plot the heatmap, expect to see replication oriented entropy
fig = plt.figure(figsize=(7, 4))

plt.subplot(1, 2, 1)
selected_repls = gene_repls.loc[lo_entropy_lo_max_ge_orfs].sort_values('replication_timing')
dat = nuc_entropy.loc[selected_repls.index]

# Some of the data is nan, so remove them, but also update the replication dataframe
# todo: clean this up
#dat = dat.dropna()
#selected_repls = selected_repls.loc[dat.index]

gene_chrom_repl_analysis.plot_heatmap(dat.values, 
                                      selected_repls, vmin=2, vmax=4,
                                     title="Nucleosome\nentropy", show_n_title=False,
    fig=fig, cmap='Spectral')
plt.colorbar()
plt.ylabel("Genes sorted by replication timing")
plt.xlabel("Cell cycle time, min")

plt.subplot(1, 2, 2)
selected_repls = gene_repls.loc[lo_entropy_lo_max_ge_orfs].sort_values('replication_timing')
dat = nuc_entropy.loc[selected_repls.index]
dat = dat - dat.mean(axis=1).values.reshape((-1, 1))

# Some of the data is nan, so remove them, but also update the replication dataframe
# todo: clean this up
#dat = dat#.dropna()
#selected_repls = selected_repls.loc[dat.index]

gene_chrom_repl_analysis.plot_heatmap(dat.values, 
                                      selected_repls, vmin=-.5, vmax=0.5,
                                     title="Normalized nucleosome\nentropy",
    fig=fig, cmap='RdBu_r', show_n_title=False)
plt.colorbar()
plt.xlabel("Cell cycle time, min")
plt.suptitle(f"Nucleosome entropy for untranscribed\ngenes over time, n={len(dat)}")
plt.subplots_adjust(top=0.75)

In [ ]:
from src.helpers import select_columns_by_indices

def plot_heatmap_delta(gene_chrom_repl_analysis, delta_in_indices):
    
    selected_repls = gene_repls.loc[lo_entropy_lo_max_ge_orfs].sort_values('replication_timing')
    
    delta_in_min = delta_in_indices/2.
    repl_indices_t = np.array(
        gene_chrom_repl_analysis.convert_h_indices_to_t(selected_repls.replication_H_index))

    repl_indices_minus = repl_indices_t-delta_in_indices
    repl_indices_plus = repl_indices_t+delta_in_indices

    # Create a heatmap of just the replication window per gene
    # Highlight the differences in chromatin for these genes.

    # Plot the heatmap, expect to see replication oriented entropy
    fig = plt.figure(figsize=FiguresConfig.FIGSIZE_SQUARE_WIDE)

    dat = nuc_entropy.loc[selected_repls.index]
    unnormalized_dat = dat.copy()
    normalized_dat = dat - dat.mean(axis=1).values.reshape((-1, 1))

    unnormalized_repl_window_dat = select_columns_by_indices(unnormalized_dat, 
        repl_indices_minus, repl_indices_plus)
    normalized_repl_window_dat = select_columns_by_indices(normalized_dat, 
        repl_indices_minus, repl_indices_plus)

    n = len(dat)

    plt.subplot(1, 2, 1)
    plt.imshow(unnormalized_repl_window_dat, cmap='Spectral', 
               vmin=2, vmax=4, aspect='auto',
              extent=[-delta_in_min, delta_in_min, 0, n],
              origin='lower', interpolation='none')
    plt.ylim(n, 0)
    plt.axvline(-10, c='black', lw=0.5, ls='solid')
    plt.axvline(0, c='black', lw=0.5, ls='dotted')
    plt.axvline(10, c='black', lw=0.5, ls='solid')
    plt.yticks([])
    plt.title("Nucleosome entropy",
              fontsize=FiguresConfig.FIG_TITLE_FONTSIZE)
    plt.colorbar()
    plt.xlabel("Min around est. replication")
    plt.ylabel("Genes sorted by replication timing")

    data_to_plot = normalized_repl_window_dat
    
    plt.subplot(1, 2, 2)
    
    n = len(data_to_plot)

    plt.imshow(data_to_plot, cmap='RdBu_r', 
               vmin=-0.5, vmax=0.5, aspect='auto',
             extent=[-delta_in_min, delta_in_min, 0, n],
              origin='lower', interpolation='none')
    plt.title("Normalized nucleosome\nentropy", 
              fontsize=FiguresConfig.FIG_TITLE_FONTSIZE)
    plt.axvline(-10, c='black', lw=0.5, ls='solid')
    plt.axvline(0, c='black', lw=0.5, ls='dotted')
    plt.axvline(10, c='black', lw=0.5, ls='solid')
    plt.ylim(n, 0)
    plt.yticks([])
    plt.colorbar()

    plt.suptitle(f"Nucleosome entropy for untranscribed\ngenes during replication, " \
                 f"n={n}", fontsize=FiguresConfig.FIG_SUPTITLE_FONTSIZE)
    plt.subplots_adjust(top=0.76)
    plt.xlabel("Min around est. replication")
    
    return fig, dat, normalized_dat, unnormalized_repl_window_dat, normalized_repl_window_dat

In [ ]:
from src.figure_configs import save_figure_for_paper
from src.plot_helpers import plot_rect2

# Delta in indices is approximately twice the length in minutes
delta_in_indices = 60
fig, dat, normalized_dat, \
    unnormalized_repl_window_dat, \
    normalized_repl_window_dat = plot_heatmap_delta(gene_chrom_repl_analysis, 
    delta_in_indices)

n = len(dat)

early_color = plt.get_cmap('plasma')(0.65)

ax = plt.gca()
plot_rect2(ax, -delta_in_min-4.75, 0.25, delta_in_min+4.75, 
    n*0.2, edgecolor=early_color, fill=None, lw=2, zorder=100)

save_figure_for_paper("output/figures/Nucleosome_entropy_replication.png")


In [ ]:
from src.helpers import get_equal_partitions

partition_indices = get_equal_partitions(normalized_repl_window_dat, k=5)


In [ ]:
from src.boxplot import get_boxplot_data_for_metric_df
from src.helpers import get_quantile_values

binstep = 5
partition_dfs = []
partition_narrow_dfs = []
for start, end in partition_indices:
    cur_partition_dat = normalized_dat.iloc[start:end]
    cur_narrow_df = get_boxplot_data_for_metric_df(cur_partition_dat, 
        cur_partition_dat.index, 'entropy', binstep=binstep, index_range_key='T_range')
    cur_narrow_df['cat_name'] = start
    partition_narrow_dfs.append(cur_narrow_df)
    partition_dfs.append(cur_partition_dat)


In [ ]:

def plot_early_df(early_df, color):

    box_plotter = BoxPlotPlotter()
    box_plotter.set_data([early_df],
                         data_key='entropy', 
                         group_key='T_range',
                         group_name_key='cat_name',
                         category_names=["Early"])
    box_plotter.ylims = None
    box_plotter.plot_outliers = False
    box_plotter.plot_whiskers = False
    box_plotter.width = 0.3
    box_plotter.legend = False
    box_plotter.color = color
    box_plotter.auto_xticks = False

    ax = plt.gca()
    title = ""
    box_plotter.plot_box_plot(ax=ax, title=title)
    plt.ylabel("")
    plt.xlabel("Min around replication")
    plt.axhline(0., lw=0.25, ls='solid', c='black')

    # Get the t range names from the replication timing
    # Use the average of the t ranges
    t_ranges_to_repl_mins = ((early_df.T_range - delta_in_indices)/2).unique()

    xticks = np.arange(0, len(t_ranges_to_repl_mins), 4)
    plt.xticks(xticks, t_ranges_to_repl_mins[xticks].astype(int))
    plt.xlim(xticks[0]-0.5, xticks[-1]+0.5)
    
    center_tick = xticks[(len(xticks)-1)//2]
    plt.axvline(center_tick, c='black', lw=0.5, ls='dotted')
    plt.axvline(center_tick+4, c='black', lw=0.5, ls='solid')
    plt.axvline(center_tick-4, c='black', lw=0.5, ls='solid')

    plt.ylabel("Normalized entropy")
    plt.ylim(-0.15, 0.15)

In [ ]:
# Concat the remaining dataframes for comparison
non_early_df = pd.concat(partition_narrow_dfs[1:])

In [ ]:


plt.figure(figsize=FiguresConfig.FIGSIZE_SQUARE_WIDE)
plt.suptitle(f"Nucleosome entropy during replication",
        fontsize=FiguresConfig.FIG_SUPTITLE_FONTSIZE)

plt.subplots_adjust(bottom=0.2, top=0.85, hspace=0.5)

plt.subplot(2, 1, 1)
plot_early_df(early_df, color=early_color)
plt.title(f"Earliest 20% replicating, n={len(early_df.orf_name.unique())}")
plt.xlabel('')

plt.subplot(2, 1, 2)
plot_early_df(non_early_df, color='#777')
plt.title(f"Non-early replicating, n={len(non_early_df.orf_name.unique())}")

save_figure_for_paper('output/figures/Early_replicating_entropy.png')


In [ ]:
early_df.orf_name.unique()